# Google Colab (Skip locally)

In [ ]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@title Link your assignment folder & install requirements
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys
import os
import shutil
import warnings

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
  warnings.warn('CUDA is not available.')

# Imports & Setup

In [ ]:
%matplotlib inline
import warnings

from main import train_model, run_attacks
from shutil import make_archive

from visualisation.AttacksVisualisation import (
    plot_attack_effect_on_accuracy,
    plot_training_curves,
    plot_transfer_attack_effects
)
from visualisation.TrainingVisualisation import (
    plot_training_result,
    plot_models_together,
)
from arguments import Arguments
from models.classical.LeNet import LeNet
from models.classical.mlp import MLP
from models.classical.resnet18 import ResNet18
from models.classical.mlpmixer import MLPMixer
from models.hybrid.HQNN_Quanv import HQNN_Quanv
from models.hybrid.HQNN_Parallel import HQNN_Parallel

from LoggingService import LoggingService

# Experiment #1

## Experiment Setup

In [ ]:
# Define a arguments and set global defaults
LoggingService("experiment1")
seeds = [0, 42]
args = Arguments()
args.verbose = False
args.n_steps = 100
args.num_workers = 0
args.train_batch_size = 512
args.eval_batch_size = 512

# 2) Define your list of model‐configurations: (name, class, kwargs)
model_specs = [
    # ("ResNet18_1", ResNet18, {
    #     "num_classes": 10
    # }),
    # ("MLPMixer_1", MLPMixer, {
    #     "num_classes": 10,
    #     "img_size": 32,
    #     "patch_size": 4,
    #     "embed_dim": 256,
    #     "num_blocks": 4,
    #     "drop_rate": 0.0,
    #     "activation": "gelu",
    #     "mlp_ratio": [1, 1]
    # }),
    # ("MLP_1", MLP, {
    #     "input_size": 3072,
    #     "hidden_sizes": [1024, 512, 64, 64],
    #     "num_classes": 10,
    #     "activation": "relu"
    # }),
    ("HQNN_Parallel_1", HQNN_Parallel, {
        "in_channels": 1,
        "classes": 10
    }),
    ("LeNet_1", LeNet, {
        "in_channels": 1,
        "classes": 10
    }),
]

## Models Training

In [ ]:
# Train each model spec across all seeds, collect results as (model, seed)
trained_models = {}
for (model_name, model_cls, config) in model_specs:
    args.exp_name = model_name
    args.model = model_cls
    args.model_config = config

    trained_models[model_name] = train_model(args, seeds)

## Attacks

In [ ]:
# Run transfer attacks
attacks_metrics = run_attacks(trained_models, args)

In [ ]:
list(trained_models.values())[0][0].all_metrics.keys()

## Figures

In [ ]:
plot_training_curves(trained_models,
                    title_prefix="training")

plot_transfer_attack_effects(attacks_metrics,
                            title_prefix="transfer")


In [ ]:
make_archive('experiment1', 'zip', 'logs/experiment1')